# Hull Tactical Market Prediction — Modeling Notebook

This notebook implements the modeling approaches described in the report:

- ElasticNet anchor model
- Ridge with top correlation feature selection
- Random Forest on core feature buckets
- PCA plus Ridge
- PLS regression
- HistGradientBoosting

All models predict next day market forward excess returns and are mapped to positions in `[0, 2]` using a volatility capped linear map. The reusable code lives in `src/pipeline.py`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from src import pipeline


In [ ]:
# Paths (adjust as needed)
DATA_DIR = Path('data/hull-tactical-market-prediction')
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'

print('Train path exists:', TRAIN_PATH.exists())
print('Test path exists:', TEST_PATH.exists())


In [ ]:
# Load data (requires the competition CSVs placed under data/hull-tactical-market-prediction)
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH.exists() else None

# Ensure time order
if 'date_id' in train.columns:
    train = train.sort_values('date_id').reset_index(drop=True)
if test is not None and 'date_id' in test.columns:
    test = test.sort_values('date_id').reset_index(drop=True)


In [ ]:
# Add lagged labels in train to match test schema
train_lag = pipeline.add_lagged_labels(train)
feature_cols_all = pipeline.get_feature_columns(train_lag, test)
X_full = train_lag[feature_cols_all].copy()
y_full = train_lag['market_forward_excess_returns'].copy()

# Drop very sparse and constant features
X_full, kept_cols = pipeline.drop_sparse_and_constant(X_full, min_frac=0.20)
X_full = X_full[kept_cols]
print('Kept features:', len(kept_cols))


## ElasticNet anchor

In [ ]:
en_model = pipeline.make_elasticnet_model(alpha=5e-4, l1_ratio=0.2)
en_trained = pipeline.train_with_cv_and_k(
    model=en_model,
    X=X_full,
    y=y_full,
    n_splits=3,
    gap=3,
    vol_cap=1.2,
    k_grid=np.linspace(0.0, 60.0, 121),
    use_isotonic=True,
)
en_trained.meta


## Ridge with top correlation feature pre selection

In [ ]:
# Compute correlations on train
corr = X_full.apply(lambda col: np.corrcoef(col.values, y_full.values)[0, 1])
corr_abs = corr.abs().sort_values(ascending=False)
top_k = 40
top_cols = corr_abs.head(top_k).index.tolist()
X_ridge = X_full[top_cols].copy()

ridge_model = pipeline.make_ridge_model(alpha=1.0)
ridge_trained = pipeline.train_with_cv_and_k(
    model=ridge_model,
    X=X_ridge,
    y=y_full,
    n_splits=3,
    gap=3,
    vol_cap=1.2,
    k_grid=np.linspace(0.0, 60.0, 121),
    use_isotonic=True,
)
ridge_trained.meta


## Random Forest on core buckets

In [ ]:
# Restrict to core feature families M*, V*, MOM*, S* and lagged labels
core_cols = []
for c in kept_cols:
    if c.startswith('M') or c.startswith('V') or c.startswith('MOM') or c.startswith('S'):
        core_cols.append(c)
    if c.startswith('lagged_'):
        core_cols.append(c)
core_cols = sorted(set(core_cols))
X_rf = X_full[core_cols].copy()

rf_model = pipeline.make_rf_model(n_estimators=200, min_samples_leaf=2, max_features='sqrt')
rf_trained = pipeline.train_with_cv_and_k(
    model=rf_model,
    X=X_rf,
    y=y_full,
    n_splits=3,
    gap=3,
    vol_cap=1.2,
    k_grid=np.linspace(0.0, 60.0, 121),
    use_isotonic=False,
)
rf_trained.meta


## PCA plus Ridge

In [ ]:
pca_ridge_model = pipeline.make_pca_ridge_model(n_components=30, alpha=1.0)
pca_ridge_trained = pipeline.train_with_cv_and_k(
    model=pca_ridge_model,
    X=X_full,
    y=y_full,
    n_splits=3,
    gap=3,
    vol_cap=1.2,
    k_grid=np.linspace(0.0, 60.0, 121),
    use_isotonic=False,
)
pca_ridge_trained.meta


## PLS regression

In [ ]:
pls_model = pipeline.make_pls_model(n_components=8)
pls_trained = pipeline.train_with_cv_and_k(
    model=pls_model,
    X=X_full,
    y=y_full,
    n_splits=3,
    gap=3,
    vol_cap=1.2,
    k_grid=np.linspace(0.0, 60.0, 121),
    use_isotonic=False,
)
pls_trained.meta


## HistGradientBoosting (HGB)

In [ ]:
hgb_model = pipeline.make_hgb_model(max_depth=3, learning_rate=0.05, max_iter=300, min_samples_leaf=20)
hgb_trained = pipeline.train_with_cv_and_k(
    model=hgb_model,
    X=X_full,
    y=y_full,
    n_splits=3,
    gap=3,
    vol_cap=1.2,
    k_grid=np.linspace(0.0, 60.0, 121),
    use_isotonic=False,
)
hgb_trained.meta


## Simple blending of mapped positions

In [ ]:
models_trained = [en_trained, ridge_trained, rf_trained, pca_ridge_trained, pls_trained, hgb_trained]

# Build positions on train for a basic check
pos_mat = []
for tm in models_trained:
    pos = pipeline.predict_positions(tm, X_full)
    pos_mat.append(pos)
pos_mat = np.vstack(pos_mat)

pos_blend = pos_mat.mean(axis=0)
pos_blend = np.clip(pos_blend, 0.0, 2.0)
ret_blend = pos_blend * y_full.values

print('Blend mean return:', float(ret_blend.mean()))
print('Blend vol ratio:', float(ret_blend.std() / (y_full.std() + 1e-12)))
